# 04 RoBERTa Pretraining

## Purpose

This notebook runs masked language model pretraining for one tokenizer setting and one model configuration.

## Inputs

- tokenizer files from one folder in `MyDrive/ProjectRoot/tokenizers/`
- tokenized datasets from one folder in `MyDrive/ProjectRoot/tokenized_datasets/`
- optional checkpoint or `best_model` folder for continuation runs

## Outputs

- training checkpoints in `MyDrive/ProjectRoot/checkpoints/<tokenizer_family>/<experiment_name>/`
- `best_model/` saved inside that experiment folder
- `trainer_state.json`
- `experiment_metadata.json`
- run index updates written to `MyDrive/ProjectRoot/registry/run_index.csv`

## Notes to myself

This is the main training notebook, so I want it to stay explicit. The two biggest things are making sure the run naming is clean and making sure continuation runs don't quietly point at the wrong tokenizer or dataset.

## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- checkpoints and heavy training artifacts stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub

In [5]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

# Mount Google Drive so the notebook can read data files and save outputs.
drive.mount('/content/drive')

# Clone the public GitHub repository into the Colab runtime.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}
!git pull origin main --no-edit -q

# Add the repo to the Python path so src/ imports work across notebooks.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository already exists. Pulling latest changes...
/content/glycan-roberta
Colab environment ready.
Repo directory: /content/glycan-roberta


## Run modes

This notebook supports three modes:

- `fresh`: start a brand-new training run
- `resume_checkpoint`: continue from a saved `checkpoint-*` folder
- `continue_best_model`: start a new continuation run from a saved `best_model` folder

The main thing to remember is:

- `resume_checkpoint` keeps the original learning-rate plan and expects total target epochs
- `continue_best_model` is a new experiment and expects only the extra continuation length

In [6]:
# ==============================================================================
# 1. DEFINE THE TRAINING CONFIGURATION
# ==============================================================================
import json
import subprocess

# --- A. RUN MODE CONTROL ---
RUN_MODE = 'continue_best_model'
# 'fresh', 'resume_checkpoint', or 'continue_best_model'

PARENT_EXPERIMENT_NAME = '/content/drive/MyDrive/ProjectRoot/checkpoints/manual/mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2'
RESUME_SOURCE_DIR = '/content/drive/MyDrive/ProjectRoot/checkpoints/manual/mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2/best_model'
# These stay empty for fresh runs. Fill them in only for continuation modes.
# Example checkpoint path:
# '/content/drive/MyDrive/ProjectRoot/checkpoints/byte_bpe/mlm15_L6_H512_A8_lr00001_ep100_setv300_m2/checkpoint-54600'
# Example best_model path:
# '/content/drive/MyDrive/ProjectRoot/checkpoints/byte_bpe/mlm15_L6_H512_A8_lr00001_ep100_setv300_m2/best_model'

# --- B. TOKENIZER AND DATASET SETTINGS ---
TOKENIZER_FAMILY = 'manual'   # 'byte_bpe', 'manual', or 'hybrid_char_bpe'
SETTING_LABEL = 'v1_train_only'               # examples: 'v300_m2', 'v1_train_only', 'v70_m2'
MLM_PROBABILITY = 0.15

# --- C. MODEL SETTINGS ---
NUM_HIDDEN_LAYERS = 6
ATTENTION_HEADS = 8
HIDDEN_SIZE = 512
INTERMEDIATE_SIZE = HIDDEN_SIZE * 4
MAX_POSITION_EMBEDDINGS = 512

# --- D. TRAINING SETTINGS ---
BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 3
EARLY_STOPPING_PATIENCE = 15
LOGGING_STEPS = 50
RANDOM_SEED = 42

# --- E. TRAINING LENGTH AND LEARNING RATE ---
INITIAL_EPOCHS = 100
CONTINUATION_EPOCHS = 20

BASE_LEARNING_RATE = 1e-4
CONTINUATION_LEARNING_RATE = 5e-5

# Convert the run mode into the effective training length and learning rate.
if RUN_MODE == 'fresh':
    EPOCHS = INITIAL_EPOCHS
    LEARNING_RATE = BASE_LEARNING_RATE
elif RUN_MODE == 'resume_checkpoint':
    if not PARENT_EXPERIMENT_NAME or not RESUME_SOURCE_DIR:
        raise ValueError('resume_checkpoint mode requires PARENT_EXPERIMENT_NAME and RESUME_SOURCE_DIR')
    EPOCHS = INITIAL_EPOCHS + CONTINUATION_EPOCHS
    LEARNING_RATE = BASE_LEARNING_RATE
elif RUN_MODE == 'continue_best_model':
    if not PARENT_EXPERIMENT_NAME or not RESUME_SOURCE_DIR:
        raise ValueError('continue_best_model mode requires PARENT_EXPERIMENT_NAME and RESUME_SOURCE_DIR')
    EPOCHS = CONTINUATION_EPOCHS
    LEARNING_RATE = CONTINUATION_LEARNING_RATE
else:
    raise ValueError(f'Unsupported RUN_MODE: {RUN_MODE}')

# Build the main Drive paths used by this run.
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot'
CHECKPOINT_ROOT = os.path.join(PROJECT_ROOT, 'checkpoints', TOKENIZER_FAMILY)
TOKENIZER_DIR = os.path.join(PROJECT_ROOT, 'tokenizers', TOKENIZER_FAMILY, SETTING_LABEL)
TOKENIZED_DATASET_DIR = os.path.join(PROJECT_ROOT, 'tokenized_datasets', TOKENIZER_FAMILY, SETTING_LABEL)
RUN_INDEX_PATH = os.path.join(PROJECT_ROOT, 'registry', 'run_index.csv')

# Make sure the tokenizer and tokenized datasets already exist before training.
for required_path in [TOKENIZER_DIR, TOKENIZED_DATASET_DIR]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f'Required path not found: {required_path}')

# Check that continuation modes point at the right kind of saved directory.
if RUN_MODE == 'resume_checkpoint':
    if not os.path.exists(RESUME_SOURCE_DIR):
        raise FileNotFoundError(f'Checkpoint not found: {RESUME_SOURCE_DIR}')
    if 'checkpoint-' not in os.path.basename(RESUME_SOURCE_DIR):
        raise ValueError('resume_checkpoint mode must point to a checkpoint-* directory')

if RUN_MODE == 'continue_best_model':
    if not os.path.exists(RESUME_SOURCE_DIR):
        raise FileNotFoundError(f'best_model directory not found: {RESUME_SOURCE_DIR}')
    if os.path.basename(RESUME_SOURCE_DIR) != 'best_model':
        raise ValueError('continue_best_model mode must point to a best_model directory')

# For continuation runs, check that the saved model architecture matches what
# this notebook is about to request.
if RUN_MODE in ['resume_checkpoint', 'continue_best_model']:
    resume_config_path = os.path.join(RESUME_SOURCE_DIR, 'config.json')
    if os.path.exists(resume_config_path):
        with open(resume_config_path, 'r', encoding='utf-8') as file:
            resume_config = json.load(file)

        expected_pairs = {
            'num_hidden_layers': NUM_HIDDEN_LAYERS,
            'num_attention_heads': ATTENTION_HEADS,
            'hidden_size': HIDDEN_SIZE,
            'intermediate_size': INTERMEDIATE_SIZE,
            'vocab_size': None,
        }

        for key, expected in expected_pairs.items():
            if key == 'vocab_size':
                continue
            observed = resume_config.get(key)
            if observed != expected:
                raise ValueError(f'Resume model mismatch for {key}: expected {expected}, found {observed}')

# Save the exact repo commit used for this run in the experiment metadata.
git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR).decode('utf-8').strip()

print('Configuration loaded.')
print(f'Run mode: {RUN_MODE}')
print(f'Tokenizer family: {TOKENIZER_FAMILY}')
print(f'Setting label: {SETTING_LABEL}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Epochs: {EPOCHS}')

Configuration loaded.
Run mode: continue_best_model
Tokenizer family: manual
Setting label: v1_train_only
Learning rate: 5e-05
Epochs: 20


## Run naming and metadata

I want the experiment folder name to carry the key training settings directly. I also want every run to register itself right away so the run index doesn't depend on me remembering to document it later.

In [7]:
# ==============================================================================
# 2. BUILD THE EXPERIMENT NAME AND REGISTER THE RUN
# ==============================================================================
from src.run_index import upsert_run_record

def format_lr_tag(value):
    return str(value).replace('.', '')

def build_base_experiment_name():
    arch_tag = f'L{NUM_HIDDEN_LAYERS}_H{HIDDEN_SIZE}_A{ATTENTION_HEADS}'
    lr_tag = format_lr_tag(LEARNING_RATE)

    # Fresh runs name the new architecture directly. Continuation modes keep
    # the parent experiment in the new folder name.
    if RUN_MODE == 'fresh':
        return f'mlm{int(MLM_PROBABILITY * 100)}_{arch_tag}_lr{lr_tag}_ep{EPOCHS}_set{SETTING_LABEL}'
    if RUN_MODE == 'resume_checkpoint':
        return f'{PARENT_EXPERIMENT_NAME}_resume_toep{EPOCHS}'
    return f'{PARENT_EXPERIMENT_NAME}_cont_lr{lr_tag}_ep{EPOCHS}'

def resolve_experiment_dir(base_dir):
    # If a folder name already exists, make a versioned copy instead of
    # overwriting an older run.
    if not os.path.exists(base_dir):
        return base_dir

    version = 2
    while True:
        candidate = f'{base_dir}_v{version}'
        if not os.path.exists(candidate):
            return candidate
        version += 1

BASE_EXPERIMENT_NAME = build_base_experiment_name()
BASE_CHECKPOINT_DIR = os.path.join(CHECKPOINT_ROOT, BASE_EXPERIMENT_NAME)
CHECKPOINT_DIR = resolve_experiment_dir(BASE_CHECKPOINT_DIR)
EXPERIMENT_NAME = os.path.basename(CHECKPOINT_DIR)
BEST_MODEL_DIR = os.path.join(CHECKPOINT_DIR, 'best_model')
TRAINER_STATE_PATH = os.path.join(CHECKPOINT_DIR, 'trainer_state.json')
LOG_DIR = os.path.join(CHECKPOINT_DIR, 'logs')
EXPERIMENT_METADATA_PATH = os.path.join(CHECKPOINT_DIR, 'experiment_metadata.json')

# Create the run folder before writing metadata or training artifacts.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Write a first-pass metadata file before training starts.
metadata_payload = {
    'experiment_name': EXPERIMENT_NAME,
    'notebook_used': 'notebooks/04_roberta_pretraining.ipynb',
    'git_commit': git_commit,
    'vault_routing': {
        'tokenizer_dir': TOKENIZER_DIR,
        'tokenized_dataset_dir': TOKENIZED_DATASET_DIR,
        'checkpoint_dir': CHECKPOINT_DIR,
        'best_model_dir': BEST_MODEL_DIR,
        'run_index_path': RUN_INDEX_PATH,
    },
    'live_hyperparameters': {
        'run_mode': RUN_MODE,
        'parent_experiment_name': PARENT_EXPERIMENT_NAME,
        'resume_source_dir': RESUME_SOURCE_DIR,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'mlm_probability': MLM_PROBABILITY,
        'num_hidden_layers': NUM_HIDDEN_LAYERS,
        'attention_heads': ATTENTION_HEADS,
        'hidden_size': HIDDEN_SIZE,
        'intermediate_size': INTERMEDIATE_SIZE,
        'max_position_embeddings': MAX_POSITION_EMBEDDINGS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'save_total_limit': SAVE_TOTAL_LIMIT,
        'random_seed': RANDOM_SEED,
        'initial_epochs': INITIAL_EPOCHS,
        'continuation_epochs': CONTINUATION_EPOCHS,
        'base_learning_rate': BASE_LEARNING_RATE,
        'continuation_learning_rate': CONTINUATION_LEARNING_RATE,
    },
    'run_status': 'configured',
}

with open(EXPERIMENT_METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata_payload, file, indent=2)

# Register the run immediately so the index records configured runs too.
upsert_run_record(
    RUN_INDEX_PATH,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'run_mode': RUN_MODE,
        'parent_experiment_name': PARENT_EXPERIMENT_NAME,
        'mlm_probability': MLM_PROBABILITY,
        'num_hidden_layers': NUM_HIDDEN_LAYERS,
        'attention_heads': ATTENTION_HEADS,
        'hidden_size': HIDDEN_SIZE,
        'intermediate_size': INTERMEDIATE_SIZE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'tokenizer_dir': TOKENIZER_DIR,
        'tokenized_dataset_dir': TOKENIZED_DATASET_DIR,
        'checkpoint_dir': CHECKPOINT_DIR,
        'results_dir': CHECKPOINT_DIR,
        'notebook_used': 'notebooks/04_roberta_pretraining.ipynb',
        'git_commit': git_commit,
        'run_status': 'configured',
        'notes': '',
    },
)

print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Run index path: {RUN_INDEX_PATH}')

Experiment name: mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2_cont_lr5e-05_ep20
Checkpoint directory: /content/drive/MyDrive/ProjectRoot/checkpoints/manual/mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2_cont_lr5e-05_ep20
Run index path: /content/drive/MyDrive/ProjectRoot/registry/run_index.csv


## Load the tokenizer

I want this separate from the config cell because it gives me a clean place to verify the vocabulary and special tokens before training starts.

In [8]:
# ==============================================================================
# 3. LOAD THE TOKENIZER
# ==============================================================================
from transformers import PreTrainedTokenizerFast

# Load the tokenizer exactly as it was saved in notebook 02.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    TOKENIZER_DIR,
    bos_token='<s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>'
)

VOCAB_SIZE = len(tokenizer)
PAD_TOKEN_ID = tokenizer.pad_token_id
MASK_TOKEN_ID = tokenizer.mask_token_id

print(f'Tokenizer loaded from: {TOKENIZER_DIR}')
print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'Pad token ID: {PAD_TOKEN_ID}')
print(f'Mask token ID: {MASK_TOKEN_ID}')

Tokenizer loaded from: /content/drive/MyDrive/ProjectRoot/tokenizers/manual/v1_train_only
Vocabulary size: 78
Pad token ID: 1
Mask token ID: 4


## Load the tokenized datasets

This notebook should only touch the train and validation splits. The test tensors should already exist from notebook 3, but they are for notebook 6, not for training decisions here.

In [9]:
# ==============================================================================
# 4. LOAD THE TOKENIZED TRAIN AND VALIDATION DATASETS
# ==============================================================================
import torch
from torch.utils.data import Dataset

# Wrap the saved tensor dictionaries so Hugging Face Trainer can iterate over them.
class GlycanDataset(Dataset):
    def __init__(self, dataset_dict):
        self.input_ids = dataset_dict['input_ids']
        self.attention_mask = dataset_dict['attention_mask']

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
        }

train_path = os.path.join(TOKENIZED_DATASET_DIR, 'train_dataset.pt')
val_path = os.path.join(TOKENIZED_DATASET_DIR, 'val_dataset.pt')
summary_path = os.path.join(TOKENIZED_DATASET_DIR, 'preprocessing_summary.json')

# Notebook 04 should only use train and validation tensors.
for required_path in [train_path, val_path]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f'Preprocessed dataset not found: {required_path}')

# Load the tokenized splits created in notebook 03.
raw_train = torch.load(train_path)
raw_val = torch.load(val_path)

train_dataset = GlycanDataset(raw_train)
val_dataset = GlycanDataset(raw_val)

train_sequence_width = int(train_dataset.input_ids.shape[1])
# Catch mismatches between tokenizer preprocessing length and model position limit.
if train_sequence_width > MAX_POSITION_EMBEDDINGS:
    raise ValueError(
        f'MAX_POSITION_EMBEDDINGS={MAX_POSITION_EMBEDDINGS} is smaller than tokenized sequence width {train_sequence_width}'
    )

preprocessing_summary = {}
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as file:
        preprocessing_summary = json.load(file)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Validation dataset size: {len(val_dataset)}')
print(f'Sequence width: {train_sequence_width}')
if preprocessing_summary:
    print(f"Selected max length from notebook 03: {preprocessing_summary.get('selected_max_length', 'not found')}")

Train dataset size: 17453
Validation dataset size: 2182
Sequence width: 56
Selected max length from notebook 03: 56


## Initialize the model

Fresh and resume-checkpoint runs start from the declared config. `continue_best_model` loads the saved best weights directly because that mode is meant to start a new experiment from a previously trained model.

In [10]:
# ==============================================================================
# 5. INITIALIZE THE MODEL
# ==============================================================================
from transformers import RobertaConfig, RobertaForMaskedLM

# Define the transformer architecture for fresh runs or checkpoint resumes.
config = RobertaConfig(
    vocab_size=VOCAB_SIZE,
    max_position_embeddings=MAX_POSITION_EMBEDDINGS,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=ATTENTION_HEADS,
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    pad_token_id=PAD_TOKEN_ID,
    type_vocab_size=1,
)

# Fresh and resume-checkpoint runs start from this declared config. Planned
# continuation runs start from a saved best_model folder instead.
if RUN_MODE in ['fresh', 'resume_checkpoint']:
    model = RobertaForMaskedLM(config)
elif RUN_MODE == 'continue_best_model':
    print(f'Loading best model weights from: {RESUME_SOURCE_DIR}')
    model = RobertaForMaskedLM.from_pretrained(RESUME_SOURCE_DIR)
else:
    raise ValueError(f'Unsupported RUN_MODE: {RUN_MODE}')

total_trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

metadata_payload['model_summary'] = {
    'total_trainable_parameters': int(total_trainable_params),
    'vocab_size': int(VOCAB_SIZE),
    'sequence_width': int(train_sequence_width),
}

with open(EXPERIMENT_METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata_payload, file, indent=2)

print(f'Total trainable parameters: {total_trainable_params:,}')

Loading best model weights from: /content/drive/MyDrive/ProjectRoot/checkpoints/manual/mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2/best_model


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Total trainable parameters: 19,481,678


## Configure training

This cell is where the masking collator and the Hugging Face training arguments get locked in. I want those choices saved into the experiment folder through the metadata file and trainer state.

In [11]:
# ==============================================================================
# 6. CONFIGURE THE DATA COLLATOR AND TRAINING ARGUMENTS
# ==============================================================================
from huggingface_hub.utils import disable_progress_bars
from transformers import DataCollatorForLanguageModeling, TrainingArguments

# Turn off Hugging Face progress bars so the notebook stays GitHub-renderable.
disable_progress_bars()
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

# Apply random masking on the fly during MLM training.
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
)

# Use epoch-level evaluation and checkpointing so later diagnostics line up with epochs.
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=LOGGING_STEPS,
    disable_tqdm=True,
    report_to='none',
    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,
    # Use mixed precision automatically when a CUDA GPU is available.
    fp16=torch.cuda.is_available(),
)

print(f'Training outputs will be saved to: {CHECKPOINT_DIR}')
print(f'fp16 enabled: {torch.cuda.is_available()}')


Training outputs will be saved to: /content/drive/MyDrive/ProjectRoot/checkpoints/manual/mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2_cont_lr5e-05_ep20
fp16 enabled: True


## Run training

This is the actual MLM training step. Right before it starts, I mark the run as `running` in the index. When it finishes, I save the best model, save the trainer state, and mark the run as `completed`.

In [12]:
# ==============================================================================
# 7. RUN MLM PRETRAINING
# ==============================================================================
from transformers import EarlyStoppingCallback, PrinterCallback, Trainer, TrainerCallback


class TrainingTableCallback(TrainerCallback):
    """Print a plain-text training table without widget progress bars."""

    def __init__(self):
        self.header = f"{'step':>8} {'epoch':>8} {'train_loss':>12} {'val_loss':>12} {'grad_norm':>12} {'lr':>12}"
        self.separator = '-' * len(self.header)
        self.started = False

    def _format_value(self, value, decimals=4, scientific=False):
        if value is None or value == '':
            return ''
        if scientific:
            return f'{float(value):.3e}'
        return f'{float(value):.{decimals}f}'

    def on_train_begin(self, args, state, control, **kwargs):
        if not self.started:
            print(self.header)
            print(self.separator)
            self.started = True

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        train_loss = logs.get('loss')
        val_loss = logs.get('eval_loss')
        grad_norm = logs.get('grad_norm')
        learning_rate = logs.get('learning_rate')
        epoch_value = logs.get('epoch', state.epoch)

        if all(value is None for value in [train_loss, val_loss, grad_norm, learning_rate]):
            return

        step_text = str(state.global_step)
        epoch_text = self._format_value(epoch_value, decimals=3) if epoch_value is not None else ''
        train_text = self._format_value(train_loss)
        val_text = self._format_value(val_loss)
        grad_text = self._format_value(grad_norm)
        lr_text = self._format_value(learning_rate, scientific=True)

        print(f'{step_text:>8} {epoch_text:>8} {train_text:>12} {val_text:>12} {grad_text:>12} {lr_text:>12}')


# Mark the run as active before the trainer starts.
metadata_payload['run_status'] = 'running'
with open(EXPERIMENT_METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata_payload, file, indent=2)

upsert_run_record(
    RUN_INDEX_PATH,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'run_mode': RUN_MODE,
        'parent_experiment_name': PARENT_EXPERIMENT_NAME,
        'mlm_probability': MLM_PROBABILITY,
        'num_hidden_layers': NUM_HIDDEN_LAYERS,
        'attention_heads': ATTENTION_HEADS,
        'hidden_size': HIDDEN_SIZE,
        'intermediate_size': INTERMEDIATE_SIZE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'tokenizer_dir': TOKENIZER_DIR,
        'tokenized_dataset_dir': TOKENIZED_DATASET_DIR,
        'checkpoint_dir': CHECKPOINT_DIR,
        'results_dir': CHECKPOINT_DIR,
        'notebook_used': 'notebooks/04_roberta_pretraining.ipynb',
        'git_commit': git_commit,
        'run_status': 'running',
        'notes': '',
    },
)

# The Trainer handles MLM masking, checkpoint saving, and validation evaluation.
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[
        TrainingTableCallback(),
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
    ],
)

# Remove the default raw-dictionary logger so the notebook stays readable.
trainer.remove_callback(PrinterCallback)

train_kwargs = {}
# Only checkpoint resumes should restore trainer state directly.
if RUN_MODE == 'resume_checkpoint':
    print(f'Resuming trainer state from checkpoint: {RESUME_SOURCE_DIR}')
    train_kwargs['resume_from_checkpoint'] = RESUME_SOURCE_DIR

# Start training and then save the selected best model into a stable folder.
trainer.train(**train_kwargs)
trainer.save_state()
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

# Final metadata and run-index update after successful completion.
metadata_payload['run_status'] = 'completed'
metadata_payload['training_artifacts'] = {
    'trainer_state_path': TRAINER_STATE_PATH,
    'best_model_dir': BEST_MODEL_DIR,
    'log_dir': LOG_DIR,
}

with open(EXPERIMENT_METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata_payload, file, indent=2)

upsert_run_record(
    RUN_INDEX_PATH,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'run_mode': RUN_MODE,
        'parent_experiment_name': PARENT_EXPERIMENT_NAME,
        'mlm_probability': MLM_PROBABILITY,
        'num_hidden_layers': NUM_HIDDEN_LAYERS,
        'attention_heads': ATTENTION_HEADS,
        'hidden_size': HIDDEN_SIZE,
        'intermediate_size': INTERMEDIATE_SIZE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'tokenizer_dir': TOKENIZER_DIR,
        'tokenized_dataset_dir': TOKENIZED_DATASET_DIR,
        'checkpoint_dir': CHECKPOINT_DIR,
        'results_dir': CHECKPOINT_DIR,
        'notebook_used': 'notebooks/04_roberta_pretraining.ipynb',
        'git_commit': git_commit,
        'run_status': 'completed',
        'notes': '',
    },
)

print('Training complete.')
print(f'Best model saved to: {BEST_MODEL_DIR}')
print(f'Trainer state saved to: {TRAINER_STATE_PATH}')


    step    epoch   train_loss     val_loss    grad_norm           lr
---------------------------------------------------------------------
      50    0.092       0.1622                    1.4542    4.978e-05
     100    0.183       0.1404                    0.8110    4.955e-05
     150    0.275       0.1528                    1.8587    4.932e-05
     200    0.366       0.1480                    0.8687    4.909e-05
     250    0.458       0.1450                    2.2742    4.886e-05
     300    0.549       0.1575                    1.9674    4.863e-05
     350    0.641       0.1661                    2.7378    4.840e-05
     400    0.733       0.1574                    2.5409    4.817e-05
     450    0.824       0.1623                    1.6134    4.794e-05
     500    0.916       0.1562                    0.9844    4.772e-05
     546    1.000                    0.1852                          


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

     550    1.007       0.1711                    1.6441    4.749e-05
     600    1.099       0.1437                    1.4892    4.726e-05
     650    1.190       0.1478                    1.3538    4.703e-05
     700    1.282       0.1778                    1.2459    4.680e-05
     750    1.374       0.1748                    1.4070    4.657e-05
     800    1.465       0.1505                    1.3895    4.634e-05
     850    1.557       0.1404                    3.1171    4.611e-05
     900    1.648       0.1628                    2.0323    4.588e-05
     950    1.740       0.1638                    1.1507    4.565e-05
    1000    1.832       0.1611                    0.8329    4.543e-05
    1050    1.923       0.1724                    1.8804    4.520e-05
    1092    2.000                    0.1903                          


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    1100    2.015       0.1575                    3.7103    4.497e-05
    1150    2.106       0.1763                    1.7709    4.474e-05
    1200    2.198       0.1479                    1.1559    4.451e-05
    1250    2.289       0.1702                    1.8162    4.428e-05
    1300    2.381       0.1570                    1.5557    4.405e-05
    1350    2.473       0.1579                    1.0551    4.382e-05
    1400    2.564       0.1646                    1.5432    4.359e-05
    1450    2.656       0.1573                    1.2130    4.337e-05
    1500    2.747       0.1399                    1.5518    4.314e-05
    1550    2.839       0.1358                    1.4689    4.291e-05
    1600    2.930       0.1532                    3.6443    4.268e-05
    1638    3.000                    0.1845                          


...

   10400   19.048       0.1383                    0.9824    2.386e-06
   10450   19.139       0.1359                    2.3617    2.157e-06
   10500   19.231       0.1259                    1.1668    1.928e-06
   10550   19.322       0.1303                    2.8201    1.699e-06
   10600   19.414       0.1315                    1.2654    1.470e-06
   10650   19.505       0.1112                    1.7297    1.241e-06
   10700   19.597       0.1255                    1.4604    1.012e-06
   10750   19.689       0.1500                    1.3494    7.830e-07
   10800   19.780       0.1405                    0.8477    5.540e-07
   10850   19.872       0.1308                    5.4634    3.251e-07
   10900   19.963       0.1306                    1.5879    9.615e-08
   10920   20.000                    0.1800                          


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete.
Best model saved to: /content/drive/MyDrive/ProjectRoot/checkpoints/manual/mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2_cont_lr5e-05_ep20/best_model
Trainer state saved to: /content/drive/MyDrive/ProjectRoot/checkpoints/manual/mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2_cont_lr5e-05_ep20/trainer_state.json
